### Toy problem for RL finetuning of LLMS

The goal of this toy project is to train a model to answer math questions correctly using RL with verifiable rewards.

**Notes**: simply whether the answer is correct. This is verifiable with no reward model needed. Reward is binary and unambiguous, no LLM-as-judge needed. Emergence of reasoning can be verified in CoT.
<!-- Simplest possible RL training loop, RLVR setup used to train reasoning models. -->

In [ ]:
!nvidia-smi

Wed Jun 24 14:44:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
!pip install trl transformers datasets accelerate -q

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import GRPOConfig, GRPOTrainer

import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully
Parameters: 494.0M


In [3]:
messages = [
    {"role": "user", "content": "What is 2 + 2? Answer with just the number."}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False
    )

response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f"Model response: {response}")

Model response: 4


### GRPO setup:
- **Task:** simple arithmetic problems
- **Reward:** Reward: 1.0 if the answer is correct, 0.0 if it is wrong
- **Goal:** observe the model learn to get answers through RL


Dataset creation

In [11]:
from datasets import Dataset
import random

# def make_math_dataset(n=200):
#     import random
#     data = []
#     for _ in range(n):
#         a = random.randint(1, 20)
#         b = random.randint(1, 20)
#         op = random.choice(["+", "-", "*"])
#         if op == "+":
#             answer = a + b
#         elif op == "-":
#             answer = a - b
#         else:
#             answer = a * b

#         prompt = f"What is {a} {op} {b}? Answer with just the number, nothing else."
#         data.append({
#             "prompt": prompt,
#             "answer": str(answer)
#         })
#     return Dataset.from_list(data)

# dataset = make_math_dataset(200)
# print(f"Dataset size: {len(dataset)}")
# print(f"Example: {dataset[0]}")
# simpler dataset
def make_math_dataset(n=200):
    data = []
    for _ in range(n):
        a = random.randint(1, 10)
        b = random.randint(1, 10)
        op = random.choice(["+", "-"])
        answer = a + b if op == "+" else a - b
        data.append({"prompt": f"What is {a} {op} {b}? Reply with only the number.", "answer": str(answer)})
    return Dataset.from_list(data)

dataset = make_math_dataset(200)

In [12]:
# def reward_fn(completions, prompts, answer, **kwargs):
#     rewards = []
#     for completion, ans in zip(completions, answer):
#         predicted = completion.strip().split()[0] if completion.strip() else ""
#         predicted = predicted.replace(".", "").replace(",", "")

#         if predicted == ans:
#             rewards.append(1.0)
#         else:
#             rewards.append(0.0)

#     return rewards

# # Quick test of the reward function
# test_completions = ["18", "18 is the answer", "17", "  18  "]
# test_answers = ["18", "18", "18", "18"]
# print(reward_fn(test_completions, None, test_answers))

# partial reward
def reward_fn(completions, prompts, answer, **kwargs):
    rewards = []
    for completion, ans in zip(completions, answer):
        completion = completion.strip()
        if not completion:
            rewards.append(0.0)
        elif completion == ans:
            rewards.append(1.0)
        elif ans in completion:  # correct answer appears somewhere
            rewards.append(0.5)
        else:
            rewards.append(0.0)
    return rewards

In [8]:
# from trl import GRPOConfig, GRPOTrainer

# training_args = GRPOConfig(
#     output_dir="./grpo-math",
#     num_train_epochs=1,
#     per_device_train_batch_size=4,
#     gradient_accumulation_steps=2,
#     learning_rate=1e-5,
#     logging_steps=10,
#     save_steps=100,
#     report_to="none",
#     max_completion_length=32,
#     num_generations=4,
# )

# trainer = GRPOTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=dataset,
#     reward_funcs=reward_fn,
#     processing_class=tokenizer,
# )

# print("Trainer set up successfully")

In [ ]:
# trainer.train()

Step,Training Loss
10,0.000000
20,0.000000
30,0.000000
40,0.000000
50,0.000000
60,0.000000
70,0.000000
80,0.000000
90,0.000000
100,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=100, training_loss=7.450580596923828e-11, metrics={'train_runtime': 315.1002, 'train_samples_per_second': 0.635, 'train_steps_per_second': 0.317, 'total_flos': 0.0, 'train_loss': 7.450580596923828e-11, 'epoch': 1.0})

In [ ]:
# def test_model(n=20):
#     import random
#     correct = 0
#     for _ in range(n):
#         a = random.randint(1, 20)
#         b = random.randint(1, 20)
#         op = random.choice(["+", "-", "*"])
#         if op == "+": answer = a + b
#         elif op == "-": answer = a - b
#         else: answer = a * b

#         messages = [{"role": "user", "content": f"What is {a} {op} {b}? Answer with just the number, nothing else."}]
#         text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#         inputs = tokenizer(text, return_tensors="pt").to(model.device)

#         with torch.no_grad():
#             outputs = model.generate(**inputs, max_new_tokens=32, do_sample=False)

#         response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
#         tokens = response.strip().split()
#         predicted = tokens[0].replace(".", "").replace(",", "") if tokens else "EMPTY"

#         is_correct = predicted == str(answer)
#         if is_correct:
#             correct += 1
#         print(f"{a} {op} {b} = {answer} | Model: '{predicted}' | {'✓' if is_correct else '✗'}")

#     print(f"\nAccuracy: {correct}/{n} = {correct/n:.0%}")

# test_model()

9 + 8 = 17 | Model: 'EMPTY' | ✗
5 * 4 = 20 | Model: 'EMPTY' | ✗
18 * 3 = 54 | Model: 'EMPTY' | ✗
14 + 2 = 16 | Model: 'EMPTY' | ✗
3 + 7 = 10 | Model: 'EMPTY' | ✗
17 + 20 = 37 | Model: 'EMPTY' | ✗
18 * 7 = 126 | Model: 'EMPTY' | ✗
18 + 14 = 32 | Model: 'EMPTY' | ✗
15 - 19 = -4 | Model: 'EMPTY' | ✗
1 * 6 = 6 | Model: 'EMPTY' | ✗
14 - 11 = 3 | Model: 'EMPTY' | ✗
5 - 7 = -2 | Model: 'EMPTY' | ✗
4 - 3 = 1 | Model: 'EMPTY' | ✗
4 - 12 = -8 | Model: 'EMPTY' | ✗
20 + 9 = 29 | Model: 'EMPTY' | ✗
15 + 18 = 33 | Model: 'EMPTY' | ✗
13 * 3 = 39 | Model: 'EMPTY' | ✗
10 - 20 = -10 | Model: 'EMPTY' | ✗
19 * 7 = 133 | Model: 'EMPTY' | ✗
3 * 2 = 6 | Model: 'EMPTY' | ✗

Accuracy: 0/20 = 0%


In [ ]:
# messages = [{"role": "user", "content": "What is 3 + 5? Answer with just the number, nothing else."}]
# text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
# inputs = tokenizer(text, return_tensors="pt").to(model.device)

# with torch.no_grad():
#     outputs = model.generate(**inputs, max_new_tokens=32, do_sample=False)

# # Show raw output including special tokens
# raw = tokenizer.decode(outputs[0], skip_special_tokens=False)
# clean = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

# print(f"Raw: {repr(raw[-200:])}")
# print(f"Clean: {repr(clean)}")

Raw: '\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nWhat is 3 + 5? Answer with just the number, nothing else.<|im_end|>\n<|im_start|>assistant\n<|endoftext|>'
Clean: ''


results:
9 + 8 = 17 | Model: 'EMPTY' | ✗
5 * 4 = 20 | Model: 'EMPTY' | ✗
18 * 3 = 54 | Model: 'EMPTY' | ✗
14 + 2 = 16 | Model: 'EMPTY' | ✗
3 + 7 = 10 | Model: 'EMPTY' | ✗
17 + 20 = 37 | Model: 'EMPTY' | ✗
18 * 7 = 126 | Model: 'EMPTY' | ✗
18 + 14 = 32 | Model: 'EMPTY' | ✗
15 - 19 = -4 | Model: 'EMPTY' | ✗
1 * 6 = 6 | Model: 'EMPTY' | ✗
14 - 11 = 3 | Model: 'EMPTY' | ✗
5 - 7 = -2 | Model: 'EMPTY' | ✗
4 - 3 = 1 | Model: 'EMPTY' | ✗
4 - 12 = -8 | Model: 'EMPTY' | ✗
20 + 9 = 29 | Model: 'EMPTY' | ✗
15 + 18 = 33 | Model: 'EMPTY' | ✗
13 * 3 = 39 | Model: 'EMPTY' | ✗
10 - 20 = -10 | Model: 'EMPTY' | ✗
19 * 7 = 133 | Model: 'EMPTY' | ✗
3 * 2 = 6 | Model: 'EMPTY' | ✗

Accuracy: 0/20 = 0%


Raw: '\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nWhat is 3 + 5? Answer with just the number, nothing else.<|im_end|>\n<|im_start|>assistant\n<|endoftext|>'
Clean: ''

Moel outputs endoftext straight after first assistant token. This is a degenerate strategy: output nothing, get consistent reward (0.0 every time, but also never "wrong" in a way that penalizes it further)

form of reward hacking

In [16]:
import inspect
sig = inspect.signature(GRPOConfig.__init__)
print([p for p in sig.parameters.keys()])

['self', 'output_dir', 'per_device_train_batch_size', 'num_train_epochs', 'max_steps', 'learning_rate', 'lr_scheduler_type', 'lr_scheduler_kwargs', 'warmup_steps', 'optim', 'optim_args', 'weight_decay', 'adam_beta1', 'adam_beta2', 'adam_epsilon', 'optim_target_modules', 'gradient_accumulation_steps', 'average_tokens_across_devices', 'max_grad_norm', 'label_smoothing_factor', 'bf16', 'fp16', 'bf16_full_eval', 'fp16_full_eval', 'tf32', 'gradient_checkpointing', 'gradient_checkpointing_kwargs', 'torch_compile', 'torch_compile_backend', 'torch_compile_mode', 'use_liger_kernel', 'liger_kernel_config', 'use_cache', 'neftune_noise_alpha', 'torch_empty_cache_steps', 'auto_find_batch_size', 'logging_strategy', 'logging_steps', 'logging_first_step', 'log_on_each_node', 'logging_nan_inf_filter', 'include_num_input_tokens_seen', 'log_level', 'log_level_replica', 'disable_tqdm', 'report_to', 'run_name', 'project', 'trackio_space_id', 'trackio_bucket_id', 'trackio_static_space_id', 'eval_strategy', 

In [13]:
# Reload fresh model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)

training_args = GRPOConfig(
    output_dir="./grpo-math-v3",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    logging_steps=10,
    save_steps=100,
    report_to="none",
    max_completion_length=16,
    num_generations=4,
    beta=0.1,
    temperature=0.7,
    log_completions=True,
    num_completions_to_print=2,
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    reward_funcs=reward_fn,
    processing_class=tokenizer,
)

trainer.train()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,0.224729
20,0.290118
30,0.286932
40,0.292959
50,0.289244
60,0.290354
70,0.292064
80,0.293335
90,0.280949
100,0.292733


╭──────────────────────────────────────────────────── Step 10 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                   ┃ Completion                               ┃ reward_fn ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ What is 5 + 6? Reply with only the       │  machine machine machine machine machine │      0.00 │      0.00 │ │
│ │ number.                                  │ machine machine machine machine machine  │           │           │ │
│ │                                          │ machine machine machine machine machine  │           │           │ │
│ │                                          │ machine                                  │           │           │ │
│ ├──────────────────────────────────────────┼──────────────────────────────────────────┼───────────┼───────────┤ │
│ │ What is 5 + 6? Reply with only the       │  machine machine machine machine machine │      0.00 │      0.00 │ │
│ │ number.                                  │ machine machine machine machine machine  │           │           │ │
│ │                                          │ machine machine machine machine machine  │           │           │ │
│ │                                          │ machine                                  │           │           │ │
│ └──────────────────────────────────────────┴──────────────────────────────────────────┴───────────┴───────────┘ │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────────── Step 90 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                   ┃ Completion                               ┃ reward_fn ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ What is 4 + 3? Reply with only the       │  machine machine machine machine machine │      0.00 │      0.00 │ │
│ │ number.                                  │ machine machine machine machine machine  │           │           │ │
│ │                                          │ machine machine machine machine machine  │           │           │ │
│ │                                          │ machine                                  │           │           │ │
│ ├──────────────────────────────────────────┼──────────────────────────────────────────┼───────────┼───────────┤ │
│ │ What is 9 - 10? Reply with only the      │  machine machine machine machine machine │      0.00 │      0.00 │ │
│ │ number.                                  │ machine machine machine machine machine  │           │           │ │
│ │                                          │ machine machine machine machine machine  │           │           │ │
│ │                                          │ machine                                  │           │           │ │
│ └──────────────────────────────────────────┴──────────────────────────────────────────┴───────────┴───────────┘ │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── Step 100 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                   ┃ Completion                               ┃ reward_fn ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ What is 6 + 2? Reply with only the       │  machine machine machine machine machine │      0.00 │      0.00 │ │
│ │ number.                                  │ machine machine machine machine machine  │           │           │ │
│ │                                          │ machine machine machine machine machine  │           │           │ │
│ │                                          │ machine                                  │           │           │ │
│ ├──────────────────────────────────────────┼──────────────────────────────────────────┼───────────┼───────────┤ │
│ │ What is 4 - 8? Reply with only the       │  machine machine machine machine machine │      0.00 │      0.00 │ │
│ │ number.                                  │ machine machine machine machine machine  │           │           │ │
│ │                                          │ machine machine machine machine machine  │           │           │ │
│ │                                          │ machine                                  │           │           │ │
│ └──────────────────────────────────────────┴──────────────────────────────────────────┴───────────┴───────────┘ │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

╭─────────────────────────────────────────────────── Step 100 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                   ┃ Completion                               ┃ reward_fn ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ What is 4 - 8? Reply with only the       │  machine machine machine machine machine │      0.00 │      0.00 │ │
│ │ number.                                  │ machine machine machine machine machine  │           │           │ │
│ │                                          │ machine machine machine machine machine  │           │           │ │
│ │                                          │ machine                                  │           │           │ │
│ ├──────────────────────────────────────────┼──────────────────────────────────────────┼───────────┼───────────┤ │
│ │ What is 6 + 2? Reply with only the       │  machine machine machine machine machine │      0.00 │      0.00 │ │
│ │ number.                                  │ machine machine machine machine machine  │           │           │ │
│ │                                          │ machine machine machine machine machine  │           │           │ │
│ │                                          │ machine                                  │           │           │ │
│ └──────────────────────────────────────────┴──────────────────────────────────────────┴───────────┴───────────┘ │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

TrainOutput(global_step=100, training_loss=0.28334158182144165, metrics={'train_runtime': 411.878, 'train_samples_per_second': 0.486, 'train_steps_per_second': 0.243, 'total_flos': 0.0, 'train_loss': 0.28334158182144165, 'epoch': 1.0})

Model collapsed again, even with partial reward approach and simpler dataset without multiplication. Presumably happens because of using the base model without proper chat template handling.
Qwen2.5-0.5B-Instruct is too small and unstable for GRPO

In [14]:
# Clear widget metadata that causes the GitHub rendering error
import IPython
IPython.display.clear_output()